# Lista 1  - Estatística Bayesiana · 2026/2

Este notebook tem como objetivo reunir os cálculos, as figuras e as tabelas correspondente a cada questão. O notebook serve como aprendizado de programação. A resolução e discussão profunda de cada questão será reservado apenas para o Overleaf, escrito por mim. 

In [2]:
# Imports necessários
from pathlib import Path                # Diretórios e nomes de arquivos.
import csv                              # Exportação de tabelas.
import numpy as np                      # Cálculos vetorizados e grades para os gráficos.
import matplotlib.pyplot as plt         # Construção das figuras.
from matplotlib.ticker import PercentFormatter, FuncFormatter
from IPython.display import display, Markdown

%config InlineBackend.figure_format = "retina"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "legend.fontsize": 9, "lines.linewidth": 2, "grid.alpha": 0.22,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 600,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
np.set_printoptions(precision=8, suppress=True)
BLUE, GOLD, GREEN, PURPLE = "#2878B5", "#B58618", "#2A8C69", "#8055A3"

BASE_DIR = Path.cwd()                    # Pasta de trabalho do kernel Jupyter.
FIG_DIR = BASE_DIR / "figuras"
TABLE_DIR = BASE_DIR / "tabelas"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print("Pasta de saída:", BASE_DIR)

def save_figure(fig, name):
    "Salva a mesma figura em PDF para o Overleaf e PNG para visualização."
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print(f"Figura salva: figuras/{name}.pdf e .png")

Pasta de saída: /home/igorstellet/PycharmProjects/Analise_Bayesiana/notebooks/listas


## Questão 1

Reprodução do exercício 1.4.1 e, além disso, realizar 3 alterações, dadas por:

- **(a)** Reduzir a taxa de falsos positivos para $0.5\%$.
- **(b)** Retomar falsos positivos de $2.3\%$ e aumentar a frequência da doença para $1/100$.
- **(c)** Retomar todos os parâmetros originais e incorporar um segundo teste positivo, com falso positivo de $0.3\%$ e falso negativo de $1.0\%$.

Em (a) e (b), a pessoa já recebeu um resultado positivo. Em (c), os dados são **dois positivos da mesma pessoa**. Não transportamos para (c) as alterações de (a) ou (b).

### Hipóteses, informações anteriores (orignais do problema) e notações
Usamos $H$ para "a pessoa tem a doença" e $\bar H$ para "não tem", essa é a afirmação que queremos comprovar com os dados (testes). O teste $j$ positivo é representado por $D_j^+$. A prior será representado por $\pi$, nesse caso é a prevalência da doença, a probabilidade de uma pessoa aleatória ter a doença. A "Likelihood" para um dado fixo é simplismente a sensbilidade do teste. O exemplo fornece (dados originais):

$$\pi=P(H\mid I)=10^{-4},\qquad s_1=P(D_1^+\mid H,I)=1-0.014=0.986,$$
$$f_1=P(D_1^+\mid\bar H,I)=0.023.$$

Aqui $s$ é a **sensibilidade**, $f$ a taxa de falso positivo.

A taxa de falso positivo $P(+\mid\bar H)$ não é a fração de pessoas sem doença entre os positivos $P(\bar H\mid+)$. Note que o espaço amostral é diferente, a taxa de falso positivo considera o espaço amostral das pessoas que tem a doença enquanto $P(\bar H\mid+)$ é o espaço amostral das pessoas que testaram positivo.


In [4]:
prevalence_original = 1 / 10_000         # Prior: frequência da doença no exemplo.
sensitivity_1 = 1 - 0.014               # Falso negativo original = 1,4%.
false_positive_1 = 0.023                # Falso positivo original = 2,3%.

def posterior_positive(prevalence, sensitivity, false_positive):
    "Retorna P(H|+) e P(+); aceita números ou arrays para construir gráficos."
    prevalence = np.asarray(prevalence, dtype=float)
    sensitivity = np.asarray(sensitivity, dtype=float)
    false_positive = np.asarray(false_positive, dtype=float)
    for probability in (prevalence, sensitivity, false_positive):
        if np.any((probability < 0) | (probability > 1)):
            raise ValueError("Probabilidades devem estar entre 0 e 1.")
    true_positive_weight = prevalence * sensitivity       # P(H,+).
    false_positive_weight = (1 - prevalence) * false_positive  # P(Hbar,+).
    evidence = true_positive_weight + false_positive_weight   # P(+).
    if np.any(evidence == 0):
        raise ValueError("O modelo atribui probabilidade zero ao dado positivo.")
    posterior = true_positive_weight / evidence           # P(H,+)/P(+).
    return posterior, evidence

p_original, evidence_original = posterior_positive(prevalence_original, sensitivity_1, false_positive_1)
print(f"Sensibilidade: {100*sensitivity_1:.2f}%")
print(f"P(+) [Evidência]: {100*evidence_original:.6f}%")
print(f"P(H|+) [Posterior], exemplo original: {100*p_original:.6f}%")

Sensibilidade: 98.60%
P(+) [Evidência]: 2.309630%
P(H|+) [Posterior], exemplo original: 0.426908%


### Evidência e fator de Bayes

A evidência, probabilidade de testar positivo, é dado pela probabilidade de testar positivo e ter a doença + a probabilidade de testar positivo e não ter a doença (somamos sobre o espaço amostral das Hipótese):

$$P(+\mid I)=\underbrace{\pi s}_{P(H,+\mid I)}+\underbrace{(1-\pi)f}_{P(\bar H,+\mid I)}.$$

Aplicando o teorema de Bayes, temos

$$\boxed{P(H\mid+,I)=\frac{\pi s}{\pi s+(1-\pi)f}.}$$

Podemos também ver como a **odds** e **fator de Bayes** se comportam:

$$O_{\rm post}=\frac{P(H\mid+)}{1-P(H\mid+)}
=\underbrace{\frac{\pi}{1-\pi}}_{O_{\rm prior}}
\underbrace{\frac{s}{f}}_{B_{H\bar H}^{(+)}}.$$

Note que, nesse caso, como não há parâmetros para marginalizar o fator de Bayes é simplismente dado pelas razões das "Likelihoods"

In [5]:
# Item (a): alteramos somente os falsos positivos.
prevalence_a = prevalence_original
false_positive_a = 0.005
p_a, evidence_a = posterior_positive(prevalence_a, sensitivity_1, false_positive_a)

# Item (b): alteramos somente a prevalência e retomamos o teste original.
prevalence_b = 1 / 100
p_b, evidence_b = posterior_positive(prevalence_b, sensitivity_1, false_positive_1)

# Guardamos os três cenários para reaproveitar nos gráficos e nas frequências naturais.
scenarios = {
    "Original": {"prevalence": prevalence_original, "fpr": false_positive_1, "color": BLUE},
    "(a)": {"prevalence": prevalence_a, "fpr": false_positive_a, "color": GOLD},
    "(b)": {"prevalence": prevalence_b, "fpr": false_positive_1, "color": GREEN},
}
print("Cenário      prevalência      falso positivo      posterior      fator de Bayes")
for name, params in scenarios.items():
    p, evidence = posterior_positive(params["prevalence"], sensitivity_1, params["fpr"])
    bayes_factor = sensitivity_1 / params["fpr"]
    prior_odds = params["prevalence"] / (1 - params["prevalence"])
    posterior_odds = prior_odds * bayes_factor
    params.update({"posterior": float(p), "evidence": float(evidence), "bayes": bayes_factor})
    assert np.isclose(p, posterior_odds / (1 + posterior_odds))
    print(f"{name:8s}     {100*params['prevalence']:8.4f}%          {100*params['fpr']:4.1f}%         {100*p:8.4f}%        {bayes_factor:8.3f}")

Cenário      prevalência      falso positivo      posterior      fator de Bayes
Original       0.0100%           2.3%           0.4269%          42.870
(a)            0.0100%           0.5%           1.9341%         197.200
(b)            1.0000%           2.3%          30.2176%          42.870


### Itens (a) e (b): o que mudou?

Em (a), a sensibilidade do teste (1-f) e a prior da população são as mesmas. A quantidade esperada de verdadeiros positivos não muda uma vez que falso negativo não muda; o que diminui é o conjunto de falsos positivos que disputa a interpretação de um resultado positivo, porém é importante notar que mesmo reduzindo em 1/5 o número de falsos positivos não aumentamos muito a posterior.

Em (b), o teste tem exatamente o mesmo fator de Bayes do caso original, ou seja, as likelihoods não mudam, o teste tem a mesma sensibilidade. Mudam as odds anteriores: uma pessoa dessa população já começa com probabilidade maior de ter a doença. **A posterior muda mesmo sem melhorar o instrumento.** O quão verdadeiramente eficaz é o teste depende fortemente da sensibilidade da prevalência da doença. Só conseguimos saber se uma dada sensibilidade de um teste é boa sabendo também o quão rara ou não é uma doença.

Vamos mostrar essas duas dependências separadamente. As linhas horizontais de 50% indicam igualdade entre as probabilidades de $H$ e $\bar H$;

In [6]:
prevalence_grid = np.geomspace(1e-6, 0.3, 900) # Eixo log: inclui doenças raras e comuns.
fpr_grid = np.geomspace(1e-6, 0.1, 900)       # Taxas de falsos positivos para o segundo painel.
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), layout="constrained")

for fpr, color, label in [(false_positive_1, BLUE, "Falso positivo = 2,3%"),
                          (false_positive_a, GOLD, "Falso positivo = 0,5%")]:
    curve, _ = posterior_positive(prevalence_grid, sensitivity_1, fpr)
    axes[0].semilogx(prevalence_grid, curve, color=color, label=label)
for prevalence, color, label in [(prevalence_original, BLUE, "Prevalência = 0,01%"),
                                (prevalence_b, GREEN, "Prevalência = 1%")]:
    curve, _ = posterior_positive(prevalence, sensitivity_1, fpr_grid)
    axes[1].semilogx(fpr_grid, curve, color=color, label=label)

for name, params in scenarios.items():
    axes[0].scatter(params["prevalence"], params["posterior"], color=params["color"], edgecolor="white", s=65, zorder=5)
    axes[1].scatter(params["fpr"], params["posterior"], color=params["color"], edgecolor="white", s=65, zorder=5)
axes[0].annotate("(b)", (prevalence_b, p_b), xytext=(8, 8), textcoords="offset points", color=GREEN)
axes[0].annotate("Original / (a)", (prevalence_original, p_a), xytext=(12, 20), textcoords="offset points", fontsize=9)
axes[1].annotate("(a)", (false_positive_a, p_a), xytext=(-16, 12), textcoords="offset points", color=GOLD)
axes[1].annotate("(b)", (false_positive_1, p_b), xytext=(-25, 12), textcoords="offset points", color=GREEN)
for ax in axes:
    ax.axhline(0.5, color="0.5", ls=":", lw=1)
    ax.set_ylim(-0.02, 1.02)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{100*value:g}%"))
    ax.set_ylabel(r"$P(H\mid +)$")
    ax.legend(loc="upper left" if ax is axes[0] else "lower left")
axes[0].set(xlabel="Prevalência (escala log)", title="Mesma sensibilidade; prevalência variável")
axes[1].set(xlabel="Taxa de falso positivo (escala log)", title="Mesma sensibilidade; falsos positivos variáveis")
save_figure(fig, "q1_ab_prevalencia_falsos_positivos")
plt.show()

Figura salva: figuras/q1_ab_prevalencia_falsos_positivos.pdf e .png


### Escala para interpretar os números


A condição $P(H\mid+)=1/2$ equivale a $\pi s=(1-\pi)f$: verdadeiros e falsos positivos têm o mesmo peso. Podemos isolar

$$\pi_{50\%}=\frac{f}{s+f},\qquad f_{50\%}=\frac{\pi s}{1-\pi}.$$

Essas expressões ajudam a entender por que reduzir $f$ de 2.3% para 0.5% ainda não torna a doença a hipótese mais provável em (a).

In [7]:
prevalence_for_half = false_positive_1 / (sensitivity_1 + false_positive_1)
fpr_for_half = prevalence_original * sensitivity_1 / (1 - prevalence_original)
print(f"Teste original: prevalência para posterior de 50% = {100*prevalence_for_half:.4f}%")
print(f"População original: falso positivo para posterior de 50% = {100*fpr_for_half:.6f}%")
print(f"O limiar em prevalência refere-se a {prevalence_for_half/prevalence_original:.1f} vezes a prevalência original.")

Teste original: prevalência para posterior de 50% = 2.2795%
População original: falso positivo para posterior de 50% = 0.009861%
O limiar em prevalência refere-se a 227.9 vezes a prevalência original.


### Analisando em população de um milhão de pessoas

Podemos simular esses dados para $N=10^6$ apenas para visualizar as proporções. 

$$\mathrm{VP}=N\pi s,\quad \mathrm{FN}=N\pi(1-s),\quad
\mathrm{FP}=N(1-\pi)f,\quad \mathrm{VN}=N(1-\pi)(1-f).$$

Entre os positivos, a fração que realmente tem a doença é $\mathrm{VP}/(\mathrm{VP}+\mathrm{FP})$, exatamente a posterio, note que o número FP está associado a prevalência, esse número que diminui a qualidade final do teste.

In [8]:
population = 1_000_000

def expected_counts(total, prevalence, sensitivity, false_positive):
    "Tabela 2x2 de contagens esperadas, sem simular uma população aleatória."
    diseased = total * prevalence
    healthy = total * (1 - prevalence)
    return {"VP": diseased * sensitivity, "FN": diseased * (1 - sensitivity),
            "FP": healthy * false_positive, "VN": healthy * (1 - false_positive)}

population_rows = []
for name, params in scenarios.items():
    counts = expected_counts(population, params["prevalence"], sensitivity_1, params["fpr"])
    params["counts"] = counts
    population_rows.extend([
        [name, "Com doença", counts["VP"], counts["FN"], population*params["prevalence"]],
        [name, "Sem doença", counts["FP"], counts["VN"], population*(1-params["prevalence"])],
    ])
    assert np.isclose(sum(counts.values()), population)
    assert np.isclose(counts["VP"]/(counts["VP"]+counts["FP"]), params["posterior"])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), layout="constrained")
positions = np.arange(len(scenarios))
vp_values = [params["counts"]["VP"] for params in scenarios.values()]
fp_values = [params["counts"]["FP"] for params in scenarios.values()]
axes[0].bar(positions-0.18, vp_values, width=0.36, color=GREEN, label="Com doença: VP")
axes[0].bar(positions+0.18, fp_values, width=0.36, color=GOLD, label="Sem doença: FP")
axes[0].set(yscale="log", ylim=(30, 100_000), ylabel="Pessoas esperadas (escala log)", title="Quem compõe os resultados positivos?")
axes[0].set_xticks(positions, scenarios.keys())
axes[0].legend()

fractions = np.array([params["posterior"] for params in scenarios.values()])
axes[1].barh(positions, fractions, color=GREEN, label="Com doença")
axes[1].barh(positions, 1-fractions, left=fractions, color=GOLD, alpha=0.75, label="Sem doença")
for y, fraction in zip(positions, fractions):
    axes[1].text(0.99, y, f"P(H|+) = {100*fraction:.2f}%", va="center", ha="right", fontsize=10)
axes[1].set_yticks(positions, scenarios.keys())
axes[1].invert_yaxis()
axes[1].set(xlim=(0, 1), xlabel="Fração dos positivos", title="Composição do grupo selecionado")
axes[1].xaxis.set_major_formatter(PercentFormatter(1))
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
save_figure(fig, "q1_ab_frequencias_naturais")
plt.show()

Figura salva: figuras/q1_ab_frequencias_naturais.pdf e .png


### Item (c): Resultados acumulados

Fazendo um novo teste sabendo o resultado do teste anterior mudamos nossa prior, pois a probabilidade de ter a doença agora é justamente dado pela posterior obtida anteriormente.

Voltamos a $\pi=10^{-4}$, $s_1=0.986$ e $f_1=0.023$. Para o segundo exame,

$$s_2=1-0.010=0.990,\qquad f_2=0.003.$$

Depois do primeiro resultado temos as novas likelihoods dadas por:

$$P(D_2^+\mid H,D_1^+,I)=P(D_2^+\mid H,I)=s_2,$$
$$P(D_2^+\mid\bar H,D_1^+,I)=P(D_2^+\mid\bar H,I)=f_2.$$

Isto é **independência condicional dado o estado de doença**, saber o resultado de um exame não influência o outro.

Sob essa hipótese, podemos atualizar em duas etapas, usando $p_1=P(H\mid D_1^+,I)$ como nova prior:

$$p_2=\frac{p_1s_2}{p_1s_2+(1-p_1)f_2}.$$

Também poderiamos pensar em usar os dois exames de uma vez:

$$\boxed{P(H\mid D_1^+,D_2^+,I)=
\frac{\pi s_1s_2}{\pi s_1s_2+(1-\pi)f_1f_2}.}$$

O denominador dessa última expressão é $P(D_1^+,D_2^+\mid I)$. Na atualização sequencial, o denominador é $P(D_2^+\mid D_1^+,I)$: são evidências de eventos diferentes.

In [9]:
sensitivity_2 = 1 - 0.010               # Falso negativo de 1,0% implica sensibilidade 99%.
false_positive_2 = 0.003               # Falso positivo de 0,3%.

# Caminho 1: a posterior do primeiro teste vira a prior do segundo.
p_c, evidence_2_given_1 = posterior_positive(p_original, sensitivity_2, false_positive_2)

# Caminho 2: construímos a probabilidade de ambos os positivos em cada hipótese.
likelihood_both_h = sensitivity_1 * sensitivity_2
likelihood_both_not_h = false_positive_1 * false_positive_2
p_c_joint, evidence_both = posterior_positive(prevalence_original, likelihood_both_h, likelihood_both_not_h)

# A ordem da atualização deve ser irrelevante sob a mesma hipótese de independência.
p_after_2_first, _ = posterior_positive(prevalence_original, sensitivity_2, false_positive_2)
p_reverse, _ = posterior_positive(p_after_2_first, sensitivity_1, false_positive_1)
assert np.allclose([p_c, p_c_joint, p_reverse], p_c)
assert np.isclose(evidence_both, evidence_original * evidence_2_given_1)

bayes_1 = sensitivity_1 / false_positive_1
bayes_2 = sensitivity_2 / false_positive_2
odds_before = prevalence_original / (1-prevalence_original)
odds_after_1 = odds_before * bayes_1
odds_after_2 = odds_after_1 * bayes_2
assert np.isclose(p_c, odds_after_2/(1+odds_after_2))

print(f"Posterior após o primeiro positivo = {100*p_original:.6f}%")
print(f"Posterior após ambos os positivos  = {100*p_c:.6f}%")
print(f"B₁ = {bayes_1:.4f}; B₂ = {bayes_2:.4f}; B₁ B₂ = {bayes_1*bayes_2:.4f}")
print(f"P(D₂+ | D₁+) = {evidence_2_given_1:.9f}")
print(f"P(D₁+, D₂+)  = {evidence_both:.9f}")

# Mantemos o número completo no cálculo principal; arredondamos só na apresentação.
p_c_rounded, _ = posterior_positive(0.0042, sensitivity_2, false_positive_2)
print(f"Se usarmos literalmente 0,42% como prior intermediária: {100*p_c_rounded:.6f}%")

Posterior após o primeiro positivo = 0.426908%
Posterior após ambos os positivos  = 58.589340%
B₁ = 42.8696; B₂ = 330.0000; B₁ B₂ = 14146.9565
P(D₂+ | D₁+) = 0.007213584
P(D₁+, D₂+)  = 0.000166607
Se usarmos literalmente 0,42% como prior intermediária: 58.191284%


### Visualizando a atualização e o fator de Bayes

In [10]:
steps = np.arange(3)
step_labels = ["Antes dos exames", "Após o primeiro +", "Após os dois +"]
probability_steps = np.array([prevalence_original, float(p_original), float(p_c)])
log_odds_steps = np.log10([odds_before, odds_after_1, odds_after_2])
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), layout="constrained")
axes[0].plot(steps, probability_steps, color=PURPLE, marker="o", ms=8)
axes[0].set(yscale="log", ylim=(4e-5, 1.3), ylabel="Probabilidade da doença (escala log)", title="Posterior depois de cada informação")
axes[0].yaxis.set_major_formatter(PercentFormatter(1, decimals=2))
axes[0].axhline(0.5, color="0.5", ls=":", lw=1)
for step, p in zip(steps, probability_steps):
    axes[0].annotate(f"{100*p:.4f}%", (step, p), xytext=(0, 10), textcoords="offset points", ha="center", fontsize=10)
axes[1].plot(steps, log_odds_steps, color=PURPLE, marker="o", ms=8)
axes[1].axhline(0, color="0.4", ls=":", lw=1, label="Odds = 1")
axes[1].annotate(f"× {bayes_1:.2f} nas odds", (0.5, np.mean(log_odds_steps[:2])), xytext=(-8, 25), textcoords="offset points", ha="center", fontsize=10)
axes[1].annotate(f"× {bayes_2:.0f} nas odds", (1.5, np.mean(log_odds_steps[1:])), xytext=(-75, 14), textcoords="offset points", fontsize=10)
axes[1].set(ylabel=r"$\log_{10} O_{H\bar H}$", title="Os fatores de Bayes multiplicam as odds", ylim=(-4.4, 0.9))
axes[1].legend(loc="lower right")
for ax in axes:
    ax.set_xticks(steps, step_labels, rotation=12)
    ax.set_xlim(-0.18, 2.18)
save_figure(fig, "q1_c_atualizacao_sequencial")
plt.show()

Figura salva: figuras/q1_c_atualizacao_sequencial.pdf e .png


### As mesmas pessoas passam pelo segundo exame

Vamos ver o efeito do segundo teste positivo na população simulada anteriormente. O segundo teste conserva 99% dos verdadeiros positivos e apenas 0.3% dos falsos positivos. Agora temos a informação do primeiro teste de que, entre os que testeram positivos, os que realmente tem a doença são apenas $0.42\%$, isso já muda o espaço amostrar e portanto a prevalência da doença.

É esse contraste que muda a composição do grupo final. Não voltamos a aplicar o segundo teste a uma população com prevalência $10^{-4}$: entre as pessoas selecionadas pelo primeiro positivo a fração com doença já é $p_1$.

In [11]:
original_counts = scenarios["Original"]["counts"]
both_diseased = original_counts["VP"] * sensitivity_2
both_healthy = original_counts["FP"] * false_positive_2
both_total = both_diseased + both_healthy
sequential_rows = [
    ["População inicial", population*prevalence_original, population*(1-prevalence_original), population, 100*prevalence_original],
    ["Primeiro positivo", original_counts["VP"], original_counts["FP"], original_counts["VP"]+original_counts["FP"], 100*float(p_original)],
    ["Dois positivos", both_diseased, both_healthy, both_total, 100*float(p_c)],
]
assert np.isclose(both_diseased / both_total, p_c)
print(f"Com doença e dois positivos: {both_diseased:.4f} pessoas esperadas")
print(f"Sem doença e dois positivos: {both_healthy:.4f} pessoas esperadas")
print(f"Fração com doença: {both_diseased:.4f} / {both_total:.4f} = {100*p_c:.4f}%")

Com doença e dois positivos: 97.6140 pessoas esperadas
Sem doença e dois positivos: 68.9931 pessoas esperadas
Fração com doença: 97.6140 / 166.6071 = 58.5893%


### Exportando as tabelas para o Overleaf

Sáidas abaixo são tabelas numéricas, com resumo do problema para apoiar minha própria redação. O arquivo `.tex` contém apenas `tabular`: posso inseri-lo em um ambiente `table` e escrever minha legenda. Os CSV preservam a precisão numérica; a apresentação usa arredondamento e vírgula decimal.

In [12]:
def export_table(name, headers, rows, decimals):
    "Mostra uma tabela e salva versões CSV e LaTeX; decimals informa as casas por coluna."
    def formatted(value, column):
        if isinstance(value, (float, int, np.floating, np.integer)):
            return f"{value:.{decimals[column]}f}".replace(".", ",")
        return str(value)
    def latex_escape(text):
        return str(text).replace("%", r"\%").replace("_", r"\_").replace("&", r"\&")
    display_rows = [[formatted(value, column) for column, value in enumerate(row)] for row in rows]
    markdown = "| " + " | ".join(headers) + " |\n| " + " | ".join([":--"]*len(headers)) + " |\n"
    markdown += "\n".join("| " + " | ".join(row) + " |" for row in display_rows)
    display(Markdown(markdown))
    with (TABLE_DIR / f"{name}.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(headers)
        writer.writerows(rows)
    align = "l" + "r"*(len(headers)-1)
    lines = [r"\begin{tabular}{"+align+"}", r"\toprule",
             " & ".join(latex_escape(h) for h in headers) + r" \\", r"\midrule"]
    lines.extend(" & ".join(latex_escape(v) for v in row) + r" \\" for row in display_rows)
    lines.extend([r"\bottomrule", r"\end{tabular}"])
    (TABLE_DIR / f"{name}.tex").write_text("\n".join(lines)+"\n", encoding="utf-8")
    print(f"Tabela salva: tabelas/{name}.tex e .csv")

result_rows = [["Original", 100*float(p_original)], ["(a)", 100*float(p_a)],
               ["(b)", 100*float(p_b)], ["(c), sem arredondamento", 100*float(p_c)],
               ["(c), partindo de 0,42%", 100*float(p_c_rounded)]]
export_table("q1_resultados", ["Cenário", "Posterior (%)"], result_rows, [0, 6])
export_table("q1_populacao", ["Cenário", "Estado", "Positivos", "Negativos", "Total"], population_rows, [0, 0, 4, 4, 0])
export_table("q1_dois_testes", ["Etapa", "Com doença", "Sem doença", "Total", "Com doença (%)"], sequential_rows, [0, 4, 4, 4, 6])

| Cenário | Posterior (%) |
| :-- | :-- |
| Original | 0,426908 |
| (a) | 1,934054 |
| (b) | 30,217591 |
| (c), sem arredondamento | 58,589340 |
| (c), partindo de 0,42% | 58,191284 |

Tabela salva: tabelas/q1_resultados.tex e .csv


| Cenário | Estado | Positivos | Negativos | Total |
| :-- | :-- | :-- | :-- | :-- |
| Original | Com doença | 98,6000 | 1,4000 | 100 |
| Original | Sem doença | 22997,7000 | 976902,3000 | 999900 |
| (a) | Com doença | 98,6000 | 1,4000 | 100 |
| (a) | Sem doença | 4999,5000 | 994900,5000 | 999900 |
| (b) | Com doença | 9860,0000 | 140,0000 | 10000 |
| (b) | Sem doença | 22770,0000 | 967230,0000 | 990000 |

Tabela salva: tabelas/q1_populacao.tex e .csv


| Etapa | Com doença | Sem doença | Total | Com doença (%) |
| :-- | :-- | :-- | :-- | :-- |
| População inicial | 100,0000 | 999900,0000 | 1000000,0000 | 0,010000 |
| Primeiro positivo | 98,6000 | 22997,7000 | 23096,3000 | 0,426908 |
| Dois positivos | 97,6140 | 68,9931 | 166,6071 | 58,589340 |

Tabela salva: tabelas/q1_dois_testes.tex e .csv


## Questão 2